# Delta Sharing (OpenSharing) - Architecture & Access Guide

This document covers how Delta Sharing works across different scenarios, including compute ownership, network access patterns, and what can/cannot be shared.

---

## Table of Contents
1. Recipient WITH a Databricks Account (Same or Different Cloud)
2. Recipient WITHOUT a Databricks Account (Open Sharing)
3. Compute Ownership & Storage Access (Serverless, Private Endpoints, Job Compute)
4. What Objects Can Be Shared & Access Limitations

## 1. Recipient WITH a Databricks Account (Databricks-to-Databricks Sharing)

### How It Works

In Databricks-to-Databricks (D2D) sharing, both the **provider** and **recipient** have Databricks workspaces with Unity Catalog metastores.

**Workflow:**
1. **Recipient** provides their Unity Catalog **sharing identifier** (global metastore ID) to the provider  
   - Format: `<cloud>:<region>:<metastore-uuid>` (e.g., `azure:eastus:12345-abcde-67890`)
2. **Provider** creates a **Share** (a named collection of data assets)
3. **Provider** creates a **Recipient** object using the sharing identifier (type = `DATABRICKS`)
4. **Provider** grants the recipient `SELECT` on the share
5. **Recipient** sees the provider in their "Shared with me" section and mounts the share as a Unity Catalog **catalog**

---

### Same Cloud Provider (e.g., Azure-to-Azure)

| Aspect | Details |
|--------|----------|
| **Protocol** | Direct data access via cloud storage credentials (cloud tokens) |
| **Network** | Traffic stays within the same cloud backbone; lower latency |
| **Egress Cost** | Minimal if in the same region; cross-region egress applies if different regions |
| **Authentication** | Unity Catalog metastore ID exchange (no tokens/credentials to manage) |
| **Access Mode** | Directory-based access — recipient gets temporary cloud credentials to read data directly from provider's storage |

---

### Different Cloud Provider (e.g., Azure-to-AWS or Azure-to-GCP)

| Aspect | Details |
|--------|----------|
| **Protocol** | Delta Sharing REST protocol over HTTPS |
| **Network** | Cross-cloud internet traffic; higher latency |
| **Egress Cost** | Cloud egress charges apply (data leaves provider's cloud) |
| **Authentication** | Same D2D flow using global metastore IDs |
| **Access Mode** | Pre-signed URLs (cloud tokens not eligible cross-cloud) |
| **Recommendation** | Share tables WITH HISTORY + CDF enabled so recipient can create a **local replica** and only pull incremental changes (reduces ongoing egress) |

---

### Key Benefits of D2D Sharing
- No credentials/tokens to manage or rotate
- Recipient accesses data like any other Unity Catalog asset
- Supports sharing of Tables, Views, Volumes, Models, and Notebooks
- Provider can share dynamic views for row/column-level security

## 2. Recipient WITHOUT a Databricks Account (Databricks-to-Open Sharing)

### How It Works

When the recipient does **not** have Databricks, the provider uses **TOKEN-based (Open) sharing**. The recipient uses any Delta Sharing-compatible client to consume the data.

**Workflow:**
1. **Provider** creates a Share and adds tables/views
2. **Provider** creates a **Recipient** with authentication type = `TOKEN`
3. Databricks generates an **activation link** (one-time use URL)
4. **Provider** sends the activation link to the recipient
5. **Recipient** downloads a **credentials file** (JSON profile containing the token + endpoint URL)
6. **Recipient** uses the credentials file with any compatible client to query the data

---

### Compatible Clients for Open Sharing Recipients

| Client | Language/Platform |
|--------|-------------------|
| `delta-sharing` Python connector | Python (pandas, Apache Spark) |
| Apache Spark with `deltasharing` format | PySpark, Scala |
| Power BI connector | Microsoft Power BI |
| Tableau connector | Tableau |
| `delta-sharing` R connector | R |
| Any REST client | Direct API calls to Delta Sharing server |

---

### Key Characteristics of Open Sharing

| Aspect | Details |
|--------|----------|
| **Authentication** | Bearer token (from credentials file) |
| **Token Rotation** | Provider can rotate tokens; old token can be expired immediately or after a grace period |
| **IP Restrictions** | Provider can assign IP access lists to the recipient for additional security |
| **Shared Assets** | Tables and Views ONLY (no Volumes, Models, or Notebooks) |
| **Access Mode** | Pre-signed URLs by default; cloud tokens (directory-based) for eligible tables |
| **Compute** | Provider's serverless compute is used for filtering/materialization |

---

### Limitations vs. D2D Sharing
- Cannot share Volumes, Models, or Notebooks
- Recipient must manage and secure the credentials file
- Tokens can expire or be rotated (operational overhead)
- Provider bears compute cost for view materialization

## 3. Compute Ownership & Storage Access

### Who Pays for Compute?``

The billing depends on **recipient compute type** and **account relationship**:

| Recipient Compute | Account Relationship | Who Pays | SKU Used | Access Method |
|-------------------|---------------------|----------|----------|----------------|
| **Serverless** | Any (same or different account) | **Recipient** | Recipient's serverless SKU | Direct access to underlying data |
| **Classic Compute** | Same account | **Recipient** | Recipient's classic SKU | Direct access to underlying data |
| **Classic Compute** | Different account | **Recipient** | Provider's interactive serverless | Provider performs filtering |
| **Open Sharing Connectors** | Any | **Provider** | Provider's interactive serverless | Provider performs filtering |

> **Key Insight:** When the recipient uses Serverless compute (even cross-account), there is **no incremental materialization charge** — the recipient reads data directly.

---

### How Recipients Access Provider's Storage Account

#### Scenario A: Recipient Using Serverless Compute

**Network Path:**
```
Recipient Workspace → Serverless Compute Plane → (Internet/Azure Backbone) → Provider's Storage Account
```

**Securing with Private Endpoints:**
- Serverless compute egress uses **stable IP addresses** from the Network Connectivity Configuration (NCC)
- Provider must **allowlist** these stable IPs on their storage account firewall
- Steps:
  1. Recipient attaches an NCC to their workspace
  2. NCC provides a set of stable egress IPs
  3. Provider adds these IPs to their Azure Storage Account firewall rules
  4. Alternatively, provider can use **Azure Private Link** with the recipient's serverless compute subnet

---

#### Scenario A-2: Azure Private Link Setup (Recipient = External Vendor, Different Company)

When the recipient is an **external vendor with their own Azure infrastructure and Databricks account**, and the provider wants to eliminate all public internet exposure to their storage, use Azure Private Link via NCC.

**Why Private Link over IP Allowlisting?**
- The vendor is a separate company — you don't control their network
- IP allowlisting still routes traffic over the public internet (just from known IPs)
- Private Link creates a **dedicated private tunnel** on Microsoft's backbone — zero public exposure
- Provider can fully disable public access on their storage account

**Architecture:**
```
┌─────────────────────────────────────────────────────────────────────────┐
│  VENDOR (Recipient) — Different Company, Own Azure Subscription         │
│                                                                         │
│  Databricks Workspace (Vendor's subscription)                           │
│       │                                                                 │
│       ▼                                                                 │
│  Serverless Compute Plane                                               │
│       │                                                                 │
│       │  (Private Endpoint — via NCC)                                   │
│       │                                                                 │
└───────┼─────────────────────────────────────────────────────────────────┘
        │
        │  ← Azure Private Backbone (NO public internet) →
        │
┌───────┼─────────────────────────────────────────────────────────────────┐
│       ▼                                                                 │
│  PROVIDER — Your Company, Your Azure Subscription                       │
│                                                                         │
│  Azure Storage Account (firewall: public access DISABLED)               │
│  └── Delta Tables (shared via OpenSharing)                              │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

**Step-by-Step Setup:**

| Step | Who Does It | Action |
|------|-------------|--------|
| 1 | **Vendor (Recipient)** | Account Admin → Databricks Account Console → Security → Network Connectivity Configurations → "Add network configuration" → Name it, select same region as workspace |
| 2 | **Vendor (Recipient)** | In the NCC → Private Endpoint Rules tab → "Add private endpoint rule" → Paste the **Azure Resource ID** of provider's storage account → Set sub-resource to `dfs` (for Delta tables) or `blob` → Status shows PENDING |
| 3 | **Vendor (Recipient)** | Account Console → Workspaces → Select workspace → Networking → Attach the NCC → Wait ~10 min |
| 4 | **Provider (You)** | Azure Portal → Your Storage Account → Networking → Private Endpoint Connections → Find the pending request from vendor's Databricks → **Approve** it |
| 5 | **Provider (You)** | (Optional) Set storage firewall "Public network access" → **Disabled** (only private endpoints can reach it now) |
| 6 | **Vendor (Recipient)** | Refresh NCC page → Confirm endpoint status = **ESTABLISHED** → Queries now work over private link |

**What the Vendor (Recipient) Needs From You (Provider):**
- The **Azure Resource ID** of your storage account
  - Format: `/subscriptions/<sub-id>/resourceGroups/<rg>/providers/Microsoft.Storage/storageAccounts/<account-name>`
  - Found in: Azure Portal → Storage Account → Overview → JSON View → Resource ID

**What You (Provider) Need to Do:**
- Approve the private endpoint connection request in Azure Portal
- Optionally disable public access on storage (strongest security posture)

**Key Constraints:**
- NCC and vendor's workspace must be in the **same Azure region**
- One NCC can serve multiple workspaces (vendor can reuse it)
- The approval is a one-time action — once ESTABLISHED, it persists
- Works with: SQL Warehouses, Jobs, Notebooks, Pipelines, Model Serving
- **Both parties remain fully independent** — no VNet peering, no shared subscriptions, no IAM cross-trust needed

**Comparison: What's Needed from Each Side**

| | IP Allowlisting | Private Link |
|---|---|---|
| **Vendor provides** | Their NCC stable IPs | Their NCC private endpoint request |
| **Provider does** | Adds IPs to storage firewall | Approves PE connection in Azure Portal |
| **Traffic path** | Public internet (from known IPs) | Microsoft private backbone |
| **Provider can disable public access?** | ❌ No (need to keep IPs open) | ✅ Yes (fully private) |
| **Ongoing maintenance** | IPs may change if NCC recreated | None once approved |

---

#### Scenario B: Recipient Using Job Compute (Classic)

**Network Path:**
```
Recipient Workspace → Classic Cluster (in recipient's VNet) → NAT Gateway/Static IP → Provider's Storage Account
```

**Securing with Private Endpoints:**
- Provider must allowlist the **egress IP** of the recipient's workspace
- The fixed egress IP can be found from:
  - NAT Gateway attached to the recipient's Azure Databricks VNet
  - Static Public IP address attached to the recipient's VNet
- Alternatively, if both are on Azure and have network peering:
  - Provider can allow the recipient's VNet to connect via **Azure Service Endpoints**
  - No internet routing needed

---

#### Scenario C: Cross-Cloud Access

**Network Path:**
```
Recipient Workspace (e.g., AWS) → Internet → Provider's Storage Account (e.g., Azure Blob)
```

- Private endpoints are NOT possible cross-cloud
- Data access is via **pre-signed URLs** over HTTPS
- Provider can:
  - Restrict recipient IPs via access lists
  - Enable CDF + history sharing so recipient creates a local replica (reduces repeated cross-cloud reads)

---

### Summary: Provider Storage Firewall Configuration

| Recipient Type | What to Allowlist on Provider's Storage |
|----------------|------------------------------------------|
| Serverless (same cloud) | Stable IPs from recipient's NCC |
| Classic Compute (same cloud) | NAT Gateway / Static Public IP of recipient's VNet |
| Same VNet / Peered VNet | Azure Service Endpoints (no public IP needed) |
| Cross-Cloud | IP-based access lists on the recipient object |
| Open Sharing (non-Databricks) | N/A — provider's serverless does the read |

## 4. What Objects Can Be Shared & Access Limitations

### Shareable Objects by Sharing Type

| Object Type | D2D Sharing (Databricks-to-Databricks) | Open Sharing (Token-based) |
|-------------|:--------------------------------------:|:--------------------------:|
| **Managed Tables** | ✅ Yes | ✅ Yes |
| **External Tables** | ✅ Yes | ✅ Yes |
| **Views** | ✅ Yes | ✅ Yes |
| **Dynamic Views** (row/column filtering) | ✅ Yes | ✅ Yes |
| **Materialized Views** | ✅ Yes | ✅ Yes |
| **Streaming Tables** | ✅ Yes | ✅ Yes |
| **Volumes** | ✅ Yes | ❌ No |
| **ML Models** | ✅ Yes | ❌ No |
| **Notebooks** | ✅ Yes | ❌ No |
| **Foreign Tables** (Lakehouse Federation) | ✅ Yes (Beta) | ✅ Yes (Beta) |

---

### Can You Share Table History?

**Yes** — Tables can be shared **WITH HISTORY** or **WITHOUT HISTORY**:

| Sharing Mode | What Recipient Gets |
|--------------|--------------------|
| **WITHOUT HISTORY** (default) | Only the latest snapshot of the table |
| **WITH HISTORY** | Full version history + ability to time travel + Change Data Feed (if enabled) |

**When shared WITH HISTORY, the recipient can:**
- Query previous versions using `VERSION AS OF` or `TIMESTAMP AS OF`
- Read **Change Data Feed (CDF)** to see row-level inserts, updates, and deletes
- Use the shared table as a **Structured Streaming** source
- Create a **local replica** by reading CDF incrementally and merging changes

**To enable history sharing:**
```sql
-- Provider side
ALTER SHARE my_share ADD TABLE catalog.schema.my_table WITH HISTORY;
```

**To enable CDF (prerequisite for incremental replication):**
```sql
-- On the source table (provider side)
ALTER TABLE catalog.schema.my_table SET TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true');
```

---

### Can the Recipient Modify Shared Data?

## **NO — Shared data is READ-ONLY**

| Operation | Allowed? | Details |
|-----------|:--------:|----------|
| **SELECT / READ** | ✅ Yes | Only permission granted is `SELECT` |
| **INSERT** | ❌ No | Recipient cannot write to provider's tables |
| **UPDATE** | ❌ No | Recipient cannot modify provider's data |
| **DELETE** | ❌ No | Recipient cannot remove provider's data |
| **DROP TABLE** | ❌ No | Recipient cannot drop provider's objects |
| **ALTER TABLE** | ❌ No | Recipient cannot change schema or properties |

> The ONLY privilege that can be granted on a share is `SELECT`. There is no write-back path via Delta Sharing.

---

### Can the Recipient Copy Data?

**Yes** — The recipient CAN create a **local copy** of the shared data:

```sql
-- Recipient side: create a local table from shared data
CREATE TABLE my_catalog.my_schema.local_copy AS
SELECT * FROM shared_catalog.shared_schema.shared_table;
```

- Once copied, the local table is fully owned by the recipient
- The local copy is independent — it does NOT stay in sync automatically
- For ongoing sync, use **CDF + Structured Streaming** or a scheduled job to pull incremental changes

---

### Recipient Data Access Summary

```
┌─────────────────────────────────────────────────────────────────┐
│                    PROVIDER SIDE                            │
│                                                             │
│  Share: "sales_share"                                       │
│  ├── Table: orders (WITH HISTORY, CDF enabled)              │
│  ├── View: revenue_summary (dynamic, filtered by recipient) │
│  ├── Volume: reference_docs (D2D only)                     │
│  └── Model: forecast_model (D2D only)                      │
│                                                             │
└─────────────────────────────────────────────────────────────────┘
                           │
                     SELECT ONLY
                           │
                           ▼
┌─────────────────────────────────────────────────────────────────┐
│                   RECIPIENT SIDE                             │
│                                                             │
│  ✅ Can: SELECT, read CDF, time travel, stream, copy data   │
│  ❌ Cannot: INSERT, UPDATE, DELETE, DROP, ALTER              │
│                                                             │
└─────────────────────────────────────────────────────────────────┘
```

---

### Additional Security Controls for Providers

| Control | Description |
|---------|-------------|
| **Dynamic Views** | Filter rows/columns based on recipient properties (e.g., `current_recipient()`) |
| **IP Access Lists** | Restrict which IPs a recipient can query from |
| **Token Rotation** | Invalidate old tokens immediately for open sharing recipients |
| **Audit Logging** | Track all recipient access via system tables |
| **Egress Monitoring** | Use `system.billing.usage` and OpenSharing materialization history tables |

## Quick Reference: Decision Matrix

### Which Sharing Type Should You Use?

```
                       Does the recipient have Databricks?
                                    │
                       ┌───────────┴───────────┐
                       │                       │
                      YES                      NO
                       │                       │
              Use D2D Sharing           Use Open (TOKEN) Sharing
              (DATABRICKS type)         (TOKEN type)
                       │                       │
              ┌───────┴───────┐               │
              │               │               │
         Same Cloud      Diff Cloud      Recipient uses:
              │               │          - Python connector
     Direct data access  Pre-signed URLs - Spark
     (cloud tokens)      + consider       - Power BI
     Low egress cost     local replica    - Tableau
                                          - REST API
```

---

### Cost Summary

| Scenario | Compute Cost Paid By | Storage/Egress Paid By |
|----------|---------------------|------------------------|
| D2D, Serverless, any account | Recipient | Provider (storage), cloud provider (egress) |
| D2D, Classic, same account | Recipient | Provider (storage) |
| D2D, Classic, different account | Recipient (via provider serverless) | Provider (storage + egress) |
| Open Sharing | Provider (serverless for materialization) | Provider (storage + egress) |

---

### Best Practices

1. **Minimize Egress**: Share tables WITH HISTORY + CDF enabled so recipients can maintain local replicas
2. **Use Dynamic Views**: For row/column-level security based on recipient identity
3. **Prefer D2D**: Whenever possible — more asset types, no token management, better security
4. **Monitor Costs**: Use `system.billing.usage` to track sharing-related compute and egress
5. **Network Security**: Use NCC (serverless) or VNet allowlisting (classic) for storage firewall rules
6. **Rotate Tokens**: For open sharing recipients, rotate tokens periodically and after any security incident